# 线性回归 学习笔记

---

## 一、原理讲解

### 1.1 线性回归在干什么

线性回归的核心任务只有一句话：**找一条直线（或高维超平面），让所有数据点到这条线的竖直距离平方和最小。**

```
原始数据是一堆 (x, y) 点。
我们假设它们之间近似满足 y ≈ θ₀ + θ₁x。
任务就是找到最好的 θ₀ 和 θ₁，让 ŷ = θ₀ + θ₁x 算出来的预测值尽量接近真实值。
```

有了这条线之后，给定一个新的 x，代入 ŷ = θ₀ + θ₁x，就能预测出 y。这就是线性回归的全部使命。

### 1.2 三个核心公式

整个算法围绕三个公式展开，它们是一条逻辑链：

**① 预测函数（假设函数）：**

```
ŷ = X · θ
```

把特征矩阵 X 乘以参数向量 θ，得到预测值。矩阵形式一行代码就能算完所有样本。

```python
predictions = np.dot(data, theta)
```

**② 损失函数（衡量预测有多烂）：**

```
J(θ) = (1/2m) · Σ(ŷ - y)²
```

每个样本的真实值 y 和预测值 ŷ 相减，平方，再求和取平均。除以 2 是为了求导时消掉平方的 2，让梯度公式更干净。

```python
delta = hypothesis(data, theta) - labels       # 误差向量, 形状 (m, 1)
cost = (1/2) * np.dot(delta.T, delta) / num_examples  # 标量
```

**③ 参数更新公式（梯度下降）：**

```
θ := θ - α · (1/m) · Xᵀ(Xθ - y)
```

这是整个算法最核心的一行。它说的是：**当前 θ 往哪个方向调，能让损失下降最快？** 答案是梯度的反方向。每次往那走一小步（步长 α），反复走，最终到达山底。

```python
theta = theta - alpha * (1/num_examples) * np.dot(delta.T, data).T
```

**三者串起来就是完整的训练过程**：用当前 θ 预测 → 算和真实值的差距 → 算每个参数该怎么调 → 调一小步 → 重复。

### 1.3 梯度下降的直觉

> 蒙着眼站在山坡上，想去山脚。每次用脚试探四周，找到最陡的下坡方向，往那走一小步。重复几百次，最终到达谷底。

- **步长 α（学习率）**：步子大小。太大可能冲过头，甚至越走越高；太小走太慢，几百步到不了山底。
- **迭代次数**：走多少步。太少还没到山底，太多浪费时间。

这跟计量经济学用正规方程 `θ = (XᵀX)⁻¹Xᵀy` 一步解出精确答案的思路完全不同。梯度下降是"反复试错逼近"，正规方程是"推导公式一步到位"。数据量巨大时正规方程算不动，梯度下降仍能工作——这是机器学习选它的原因。

---

## 二、实操讲解

### 2.1 整体架构：三个文件，一条数据流

```
项目结构：
ML_LinearRegression/
├── data/world-happiness-report-2017.csv    ← 原始数据
├── utils/features.py                       ← 预处理函数
├── linear_regression.py                    ← 算法类
└── 调用脚本.py                              ← 训练 + 预测 + 画图
```

**数据从头到尾的流动：**

```
CSV 加载 → 切分 80% 训练 / 20% 测试
                ↓
训练集 X_train ──→ prepare_for_training() ──→ 归一化 + 加常数项
                                                    ↓
                                              self.data（训练用）
                                              self.mean, self.deviation（存起来）
                                                    ↓
                                              train() → 梯度下降 500 次
                                                    ↓
                                              self.theta（训练完成）
                                                    ↓
                         ┌──────────────────────────┘
                         ↓                          ↓
                predict(测试集 X_test)        predict(linspace 人造点)
                用 self.mean/std 预处理         用 self.mean/std 预处理
                         ↓                          ↓
                    算 R² 评估模型              画红色回归线
```

### 2.2 调用脚本：每一行在做什么

```python
# ① 加载数据
data = pd.read_csv('data/world-happiness-report-2017.csv')

# ② 切分训练集和测试集 —— 必须在预处理之前切！
train_data = data.sample(frac=0.8, random_state=42)   # 随机抽 80%
test_data = data.drop(train_data.index)                # 剩下 20%

# ③ 提取为 numpy 二维数组（双中括号保证形状是 (m,1) 不是 (m,)）
X_train = train_data[['Economy..GDP.per.Capita.']].values
y_train = train_data[['Happiness.Score']].values
X_test  = test_data[['Economy..GDP.per.Capita.']].values
y_test  = test_data[['Happiness.Score']].values

# ④ 创建模型 —— __init__ 内部自动做了预处理
linearregression = LinearRegression(X_train, y_train)
# 此时 self.data 已归一化、加了常数项，self.theta 全是零

# ⑤ 训练 —— 梯度下降 500 次
(theta, cost_history) = linearregression.train(learning_rate=0.01, num_iterations=500)

# ⑥ 画损失下降曲线 —— 观察是否收敛
plt.plot(range(num_iterations), cost_history)
```

**为什么要先切分再预处理？** 因为归一化的 mean 和 std 只能从训练集算。如果先对全量数据归一化再切分，测试集的信息就泄露到训练中了——测试集的数值参与了 mean/std 的计算。这是机器学习最核心的纪律之一。

### 2.3 画回归线的原理

```python
# 生成 100 个均匀分布的人造 x 点，只在训练集范围内
x_predictions = np.linspace(X_train.min(), X_train.max(), 100).reshape(100, 1)

# 用训练好的 θ 对这 100 个点做预测
y_predictions = linearregression.predict(x_predictions)

# 连线就是红色回归线
plt.plot(x_predictions, y_predictions, color='red')
```

**这 100 个点不是训练集，也不是测试集。** 它们是 `linspace` 均匀生成的"道具"，唯一作用是让回归线画出来平滑连续。回归线展示的是模型从训练集学到的"GDP → 幸福指数"这条规律。

**图中三种元素的区别：**

| 图形 | 数据来源 | 角色 |
|------|---------|------|
| 🟠 橙色散点 | X_train, y_train（真实数据） | 模型学习用的样本 |
| 🔵 蓝色散点 | X_test, y_test（真实数据） | 模型没见过的样本，用于检验 |
| 🔴 红色回归线 | linspace 人造点 + 模型预测值 | 模型学到的规律可视化 |

### 2.4 算法类：逐方法讲解

#### `__init__` —— 模型的出生设定

```python
def __init__(self, data, labels, polynomial_degree=0, sinusoid_degree=0, normalize_data=True):
    (data_processed, features_mean, features_deviation) = \
        prepare_for_training(data, polynomial_degree, sinusoid_degree, normalize_data)

    self.data = data_processed           # 处理后的训练数据（归一化 + 常数项）
    self.labels = labels                 # 原始训练标签
    self.mean = features_mean            # ★ 训练集的均值，留给 predict 用
    self.deviation = features_deviation  # ★ 训练集的标准差，留给 predict 用
    self.polynomial_degree = polynomial_degree
    self.sinusoid_degree = sinusoid_degree
    self.normalize_data = normalize_data
    num_features = self.data.shape[1]    # 预处理后的特征列数
    self.theta = np.zeros((num_features, 1))  # θ 初始化为全零
```

做了三件事：**预处理数据 → 把结果存进 self → 初始化 θ 为零。** `self.mean` 和 `self.deviation` 存下来的意义在于——后续 `predict` 时必须用同一套参数归一化新数据。

#### `train` → `gradient_descent` → `gradient_step` —— 训练的三层调用

**train（入口）：**

```python
def train(self, alpha, num_iterations=500):
    cost_history = self.gradient_descent(alpha, num_iterations)
    return self.theta, cost_history
```

把活交给 `gradient_descent`，返回最终的 θ 和损失记录。返回 θ 不是内部需要（`self.theta` 对象自己能读），而是给外部调用方一个方便。

**gradient_descent（循环控制）：**

```python
def gradient_descent(self, alpha, num_iterations=500):
    cost_history = []
    for _ in range(num_iterations):
        self.gradient_step(alpha)                              # 更新一步
        cost_history.append(self.cost_function(self.data, self.labels))  # 记录损失
    return cost_history
```

负责"走多少步、每一步后记录什么"。不关心步子里面的公式长什么样——那是 `gradient_step` 的事。分开写的目的是：想加早停或换优化器时只改这一层，不用动梯度公式。

**gradient_step（核心——最需要会默写的方法）：**

```python
def gradient_step(self, alpha):
    num_examples = self.data.shape[0]                           # m
    prediction = LinearRegression.hypothesis(self.data, self.theta)  # Xθ, (m,1)
    delta = prediction - self.labels                            # 误差, (m,1)
    theta = self.theta
    theta = theta - alpha * (1/num_examples) * np.dot(delta.T, self.data).T
    self.theta = theta                                          # ★ 写回去，不能忘！
```

**为什么是 `np.dot(delta.T, data).T`？** 拆开形状看：

```
delta.T @ data = (1,m) × (m,n) = (1,n)    ← 每个特征上的误差累积
转置 → (n,1)                               ← 变成列向量，和 θ 形状一致
```

这行代码的本质是：**用误差向量去"追问"每个特征——"这个特征对误差贡献了多少？"** 然后把每个 θ 往减少贡献的方向调。最后 `self.theta = theta` 写回对象，下一次循环读到更新后的值。

#### `predict` 和 `get_cost` —— 预测与评估

```python
def predict(self, data):
    data_processed = prepare_for_training(
        data,
        self.polynomial_degree,
        self.sinusoid_degree,
        self.normalize_data,
        features_mean=self.mean,            # ★ 训练时的均值
        features_deviation=self.deviation   # ★ 训练时的标准差
    )[0]
    return LinearRegression.hypothesis(data_processed, self.theta)
```

`predict` 内部**重新调了 `prepare_for_training`**，但这次传入了 `self.mean` 和 `self.deviation`。这意味着新数据用训练时的参数归一化，θ 在同一个坐标系下工作。`get_cost` 逻辑完全相同，只是最后调用 `cost_function` 而不是 `hypothesis`。

### 2.5 预处理函数：prepare_for_training

```python
def prepare_for_training(data, polynomial_degree=0, sinusoid_degree=0,
                         normalize_data=True,
                         features_mean=None,           # ← 预测时传入
                         features_deviation=None):     # ← 预测时传入

    num_examples = data.shape[0]
    data_processed = np.copy(data)

    # 归一化
    if normalize_data:
        if features_mean is not None and features_deviation is not None:
            # 预测时：用外部传入的训练集参数，不重新算
            data_normalized = (data_processed - features_mean) / features_deviation
        else:
            # 训练时：自己算 mean/std
            data_normalized, features_mean, features_deviation = normalize(data_processed)
        data_processed = data_normalized

    # 可选：正弦特征、多项式特征
    # ...

    # 最后：加一列 1（对应截距 θ₀）
    data_processed = np.hstack((np.ones((num_examples, 1)), data_processed))

    return data_processed, features_mean, features_deviation
```

**处理顺序：归一化 → 可选特征 → 加常数项 1。** 常数项永远在最后加，所以它不参与前面的归一化和特征变换。这个函数的设计核心就是那个 `if-else`——**训练时自己算参数，预测时用传入的参数**，一套逻辑，两种模式。

### 2.6 形状链路（写代码时对照）

```
X_train:        (m, 1)    ← 只有一列原始特征
预处理后:       (m, 2)    ← 加了第 0 列全 1
self.theta:     (2, 1)    ← θ₀（截距）+ θ₁（斜率）
预测输入:       (k, 2)    ← 新数据同样预处理
预测输出:       (k, 1)    ← ŷ

核心规则：θ 的行数 = 预处理后数据的列数。
```

### 2.7 评估模型

回归不能数"对了几个"，而是量"预测值和真实值差多远"。

```python
y_pred = model.predict(X_test)

# R²：1 完美，0 跟瞎猜一样，<0 模型废了
ss_res = np.sum((y_test - y_pred) ** 2)          # 模型残差
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)  # 瞎猜（用均值）的残差
r2 = 1 - (ss_res / ss_tot)

# RMSE：单位跟原始数据一致，直观
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
```

**训练损失 vs 测试损失必须对比着看：**

| 训练损失 | 测试损失 | 诊断 |
|---------|---------|------|
| 高 | 高 | 欠拟合，模型太简单 |
| 低 | 低（接近） | ✅ 泛化能力好 |
| 低 | 高（差很多） | ❌ 过拟合，背答案了 |

---

## 三、问题讲解

### 3.1 归一化泄漏——最隐蔽的 bug

**症状**：训练损失很低，但一预测新数据就全错。

**根源**：`predict` 里调了 `prepare_for_training(data)` 没传 mean/std，函数从新数据重新算了一套归一化参数。θ 是在 `(x - mean_train) / std_train` 空间训练的，预测却换到了 `(x - mean_new) / std_new` 空间。等于让一个在中国学的地图的人去美国导航——坐标系全变了。

**修复（方案A）**：给 `prepare_for_training` 加两个可选参数 `features_mean` 和 `features_deviation`。训练时不传（函数自己算），预测时传入 `self.mean` 和 `self.deviation`。函数里用一个 `if-else` 判断走哪条分支。**改动只涉及一个函数，不重复逻辑。**

```python
# predict 的正确调用方式：
data_processed = prepare_for_training(
    data, ...,
    features_mean=self.mean,            # 不能漏
    features_deviation=self.deviation   # 不能漏
)[0]
```

### 3.2 cost_function 的 self.data 陷阱

**原代码（有 bug）：**
```python
delta = LinearRegression.hypothesis(self.data, self.theta) - labels
```

传进来的参数叫 `data`，内部却用了 `self.data`（永远是训练集）。`get_cost(X_test, y_test)` 传入测试集，结果算的还是训练集损失——测试集评估完全失效，两个损失永远一样。

**修复后：**
```python
delta = LinearRegression.hypothesis(data, self.theta) - labels
```

传入什么就评估什么。这是最容易被忽略的 bug，因为代码能跑、不报错，只是结果悄悄错。

### 3.3 面向对象设计的三个易错点

**① gradient_step 最后忘了写回 `self.theta`**

```python
theta = self.theta
theta = theta - alpha * ... * np.dot(delta.T, self.data).T
self.theta = theta   # ← 这行不能漏！漏了等于 500 轮什么都没改
```

局部变量 `theta` 算对了，但没写回 `self.theta`，下一次循环读的还是旧值。

**② `self.xxx` 和参数的混淆**

一个简单判断：这个值会被多个方法用到吗？

- 是 → `self.xxx`（如 `self.theta`、`self.mean`，跨越 `__init__`、`train`、`predict`）
- 否 → 参数，用完就扔（如 `alpha`、`num_iterations`）

**③ `np.dot` 返回矩阵不是标量**

```python
cost = (1/2) * np.dot(delta.T, delta) / num_examples
# cost 是形状 (1,1) 的矩阵，不是数字
return cost[0][0]   # 从矩阵里取出标量
```

不取 `[0][0]`，后面画图或打印可能报错或显示异常。

### 3.4 数据形状的常见错误

```python
# ❌ 单中括号得到一维数组 (m,)，不是 (m,1)
X_train = train_data['Economy..GDP.per.Capita.'].values

# ✅ 双中括号得到二维数组 (m,1)
X_train = train_data[['Economy..GDP.per.Capita.']].values
```

线性回归的所有矩阵运算都假设输入是二维数组。形状不对，`np.dot` 会静默产生错误结果而不是报错。

### 3.5 梯度下降 vs 正规方程

| | 梯度下降（本代码） | 正规方程（计量经济学） |
|--|------------------|---------------------|
| 公式 | `θ := θ - α·∇J`（迭代） | `θ = (XᵀX)⁻¹Xᵀy`（一步） |
| 需要选 α 吗 | 需要 | 不需要 |
| 需要选迭代次数吗 | 需要 | 不需要 |
| 大数据量 | ✅ 能跑 | ❌ 矩阵求逆太慢 |
| 小数据量 | 偏慢 | ✅ 一步到位 |

两者数学上等价，只是求解路径不同。你的代码走的是机器学习路线（梯度下降），Stata 走的是计量经济学路线（正规方程）。

---

## 四、自我检验

以下问题能不看笔记回答，就代表真正学会了：

1. `gradient_step` 里 `np.dot(delta.T, data).T` 每一步的形状变化？为什么最后要转置？
2. 为什么 `predict` 必须传 `self.mean`？不传的后果是什么？
3. `cost_function` 的参数 `data` 和 `self.data` 各自在什么场景使用？
4. `self.theta` 的行数由什么决定？和数据预处理有什么关系？
5. 训练集、linspace 人造点、测试集分别承担什么角色？哪一个绝对不能参与任何参数计算？

---

## 五、下一步

- [ ] 用测试集评估模型（`predict(X_test)` + 计算 R²）
- [ ] 试试 `polynomial_degree=2`，观察回归线从直线变曲线
- [ ] 换不同学习率（0.001 / 0.1），看损失曲线是收敛还是震荡
- [ ] 用两个特征（GDP + 健康指数）训练，理解多维 θ 的含义
- [ ] 学习下一个算法：逻辑回归（分类的 baseline）
